# Trendline Family Research Lab

Canonical Phase A-I geometry research console. Research-only.

## 0. Scope, Safety, and Execution Mode

Uses canonical family contracts only. No regime analysis, runtime promotion, trading policy, or configuration mutation. Exact rails, corridors, interaction zones, and uncertainty remain different concepts. Default run uses offline smoke data. Holdout stays closed.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').is_file())
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

SMOKE_MODE = True
FETCH_REMOTE = False
RUN_POINT_IN_TIME_REPLAY = False
RUN_MTF_RESEARCH = False
RUN_MULTI_ASSET_COMPARISON = False
EXPORT_ARTIFACTS = False

EXECUTION_FLAGS = {
    'smoke_mode': SMOKE_MODE, 'fetch_remote': FETCH_REMOTE,
    'run_point_in_time_replay': RUN_POINT_IN_TIME_REPLAY, 'run_mtf_research': RUN_MTF_RESEARCH,
    'run_multi_asset_comparison': RUN_MULTI_ASSET_COMPARISON, 'export_artifacts': EXPORT_ARTIFACTS,
}

## 1. Research Configuration

In [ ]:
import pandas as pd
from libs.models.trendline_family import MTFNormalizationContext, compose_trendline_family_mtf
from libs.models.trendline_family.research_lab import (
    CrossAssetComparabilityPolicy, artifact_trial_rows, audit_cross_asset_comparability,
    build_cross_asset_comparison,
    build_smoke_config, build_smoke_ohlcv,
    candidate_rows, candidate_status_rows, corridor_rows, dataset_summary, event_rows,
    export_research_artifacts, family_lineage_rows, family_rows, immutable_research_frame,
    event_transition_rows, interaction_zone_rows, load_local_ohlcv, load_verified_phase_i_artifacts,
    member_rail_rows, normalize_binance_ohlcv,
    mtf_cluster_rows, mtf_projected_family_rows, mtf_projected_member_rows, mtf_relation_rows,
    mtf_source_rows, observation_rows, provider_audit_rows, record_to_dict,
    replay_corridor_rows, replay_event_rows, replay_event_transition_rows, replay_family_rows,
    replay_member_rail_rows, replay_observation_rows,
    replay_prefix_is_causal, replay_transition_rows, run_canonical_replay, snapshot_summary,
    source_group_audit_rows, structural_outcome_rows, transition_rows, validate_research_config,
    validate_research_mode,
)

ASSET = 'BTCUSDT'
TIMEFRAME = '1h'
START = None
END = None
LOCAL_DATA_PATH = None
RESOLVED_CONFIG = None
MTF_RESOLVED_CONFIG = None
PHASE_I_ARTIFACT_ROOT = None
MTF_SOURCE_SNAPSHOTS = None
MTF_NORMALIZATION_CONTEXT = None
MULTI_ASSET_REPLAYS = ()
COMPARISON_POLICY = CrossAssetComparabilityPolicy()
OUTPUT_ROOT = PROJECT_ROOT / 'artifacts' / 'trendline_family_research_lab'
CHART_LOOKBACK = 80
REPLAY_END_POSITION = None

validate_research_mode(
    smoke_mode=SMOKE_MODE, fetch_remote=FETCH_REMOTE, local_data_path=LOCAL_DATA_PATH,
    start=START, end=END,
)
resolved_config = build_smoke_config(asset=ASSET, timeframe=TIMEFRAME) if SMOKE_MODE else validate_research_config(RESOLVED_CONFIG, asset=ASSET, timeframe=TIMEFRAME)
display({
    'model_version': resolved_config.model_version,
    'config_version': resolved_config.config_version,
    'tracking_config_hash': resolved_config.resolved_config_hash,
    'mtf_config_hash': resolved_config.mtf_config_hash,
})

## 2. Data Loading and Validation

In [ ]:
if SMOKE_MODE:
    raw_ohlcv = build_smoke_ohlcv()
elif LOCAL_DATA_PATH:
    raw_ohlcv = load_local_ohlcv(LOCAL_DATA_PATH)
else:
    raw_ohlcv = None

if raw_ohlcv is not None:
    dataset = immutable_research_frame(frame=raw_ohlcv, asset=ASSET, timeframe=TIMEFRAME)
    display(record_to_dict(dataset_summary(dataset)))

In [ ]:
# Explicit remote action. Never runs in smoke mode or tests.
if FETCH_REMOTE:
    from apps.ingestion_app.adapters.binance_native import BinanceNativeAdapter

    if START is None or END is None:
        raise RuntimeError('FETCH_REMOTE requires UTC START and END.')
    adapter = BinanceNativeAdapter()
    remote = await adapter.get_historical_ohlcv(
        ASSET, TIMEFRAME,
        since=int(pd.Timestamp(START).timestamp() * 1000),
        until=int(pd.Timestamp(END).timestamp() * 1000),
    )
    raw_ohlcv = normalize_binance_ohlcv(
        remote, timeframe=TIMEFRAME, closed_before=pd.Timestamp(END).to_pydatetime(),
    )
    dataset = immutable_research_frame(frame=raw_ohlcv, asset=ASSET, timeframe=TIMEFRAME)
    display(record_to_dict(dataset_summary(dataset)))

if not FETCH_REMOTE:
    assert 'dataset' in globals()

## 3. Canonical Single-Timeframe Replay

In [ ]:
replay = run_canonical_replay(
    dataset=dataset,
    config=resolved_config,
    research_parameters={'smoke_mode': SMOKE_MODE, 'chart_lookback': CHART_LOOKBACK},
)
final_snapshot = replay.outputs[-1].snapshot
display(record_to_dict(snapshot_summary(final_snapshot)))

## 4. Price, Exact Rails, Corridors, and Interaction Zones

In [ ]:
display({
    'visualization_status': 'RETIRED',
    'replacement': 'apps.trendline_v2_viewer',
    'reason': 'Trendline visualization is owned by the TVLC-based Trendline V2 viewer.',
})

## 5. Candidate and Pivot Diagnostics

In [ ]:
candidate_table = pd.DataFrame([record_to_dict(row) for row in candidate_rows(replay)])
candidate_status_table = pd.DataFrame([record_to_dict(row) for row in candidate_status_rows(replay)])
provider_audit_table = pd.DataFrame([record_to_dict(row) for row in provider_audit_rows(replay)])
display(candidate_status_table.groupby('status', dropna=False).agg(candidate_count=('candidate_count', 'sum'), bars=('status', 'size')))
display(candidate_table.head(20))
display(provider_audit_table)
# Any score shown here is display-only and comes from persisted diagnostics or Phase-I metrics.

## 6. Family Tracker Diagnostics

In [ ]:
family_table = pd.DataFrame([record_to_dict(row) for row in family_rows(final_snapshot)])
transition_table = pd.DataFrame([record_to_dict(row) for row in replay_transition_rows(replay)])
family_history_table = pd.DataFrame([record_to_dict(row) for row in replay_family_rows(replay)])
member_history_table = pd.DataFrame([record_to_dict(row) for row in replay_member_rail_rows(replay)])
corridor_history_table = pd.DataFrame([record_to_dict(row) for row in replay_corridor_rows(replay)])
source_group_table = pd.DataFrame([record_to_dict(row) for output in replay.outputs for row in source_group_audit_rows(output.snapshot)])
display(family_table)
display(transition_table)
display(family_history_table)
display(member_history_table)
display(corridor_history_table)
display(source_group_table)

def selected_family_lineage(family_id):
    return pd.DataFrame([record_to_dict(row) for row in family_lineage_rows(replay, family_id=family_id)])

## 7. Interaction and Event Lifecycle Diagnostics

In [ ]:
observation_table = pd.DataFrame([record_to_dict(row) for row in replay_observation_rows(replay)])
event_table = pd.DataFrame([record_to_dict(row) for row in replay_event_rows(replay)])
event_transition_table = pd.DataFrame([record_to_dict(row) for row in replay_event_transition_rows(replay)])
display(observation_table)
display(event_table)
display(event_transition_table)

## 8. Point-in-Time Replay Evidence

In [ ]:
def render_replay_step(position):
    output = replay.output_at(position)
    snapshot = output.snapshot
    return {
        'summary': record_to_dict(snapshot_summary(snapshot)),
        'transitions': [record_to_dict(row) for row in transition_rows(snapshot)],
        'observations': [record_to_dict(row) for row in observation_rows(snapshot)],
        'events': [record_to_dict(row) for row in event_rows(snapshot)],
    }

if RUN_POINT_IN_TIME_REPLAY:
    replay_position = dataset.row_count - 1 if REPLAY_END_POSITION is None else REPLAY_END_POSITION
    step_evidence = render_replay_step(replay_position)
    display(step_evidence)

assert replay_prefix_is_causal(replay, position=min(20, dataset.row_count - 1), config=resolved_config)

## 9. Longevity and Structural Outcome Analysis

Persisted structural history only. Forward candidate/event outcomes stay unavailable until a reviewed Phase-I policy evaluation. No PnL or probability claims.

In [ ]:
structural_outcome_table = pd.DataFrame([record_to_dict(row) for row in structural_outcome_rows(replay)])
display(structural_outcome_table)
display({'forward_outcomes': 'UNAVAILABLE', 'reason': 'requires reviewed Phase-I frozen-stream evaluation; notebook does not open holdout'})

## 10. Multi-Timeframe Geometry Evidence

In [ ]:
def compose_mtf_research(source_snapshots, decision_timestamp, normalization_context):
    return compose_trendline_family_mtf(
        source_snapshots=source_snapshots,
        decision_timestamp=decision_timestamp,
        normalization_context=normalization_context,
        config=validate_research_config(MTF_RESOLVED_CONFIG or resolved_config, asset=ASSET, timeframe=TIMEFRAME, require_mtf=True),
    )

def mtf_research_tables(mtf_snapshot):
    return {
        'sources': pd.DataFrame([record_to_dict(row) for row in mtf_source_rows(mtf_snapshot)]),
        'projected_families': pd.DataFrame([record_to_dict(row) for row in mtf_projected_family_rows(mtf_snapshot)]),
        'projected_members': pd.DataFrame([record_to_dict(row) for row in mtf_projected_member_rows(mtf_snapshot)]),
        'relations': pd.DataFrame([record_to_dict(row) for row in mtf_relation_rows(mtf_snapshot)]),
        'clusters': pd.DataFrame([record_to_dict(row) for row in mtf_cluster_rows(mtf_snapshot)]),
    }

def render_mtf_research(source_snapshots, decision_timestamp, normalization_context):
    mtf_snapshot = compose_mtf_research(source_snapshots, decision_timestamp, normalization_context)
    return mtf_snapshot, mtf_research_tables(mtf_snapshot)

mtf_snapshot = None
if RUN_MTF_RESEARCH:
    if MTF_SOURCE_SNAPSHOTS is None or MTF_NORMALIZATION_CONTEXT is None:
        display({'status': 'UNAVAILABLE', 'reason': 'supply independently replayed confirmed source snapshots and normalization context'})
    else:
        mtf_snapshot, mtf_tables = render_mtf_research(MTF_SOURCE_SNAPSHOTS, dataset.timestamps[-1], MTF_NORMALIZATION_CONTEXT)
        for name, table in mtf_tables.items():
            display({name: table})
# Tables expose persisted projected structures and source audits; they do not refit or average geometry.

## 11. Phase-I Artifact Browser

In [ ]:
phase_i_browser = None
if PHASE_I_ARTIFACT_ROOT:
    phase_i_browser = load_verified_phase_i_artifacts(PHASE_I_ARTIFACT_ROOT)
    artifact_trial_table = pd.DataFrame([record_to_dict(row) for row in artifact_trial_rows(phase_i_browser.trials)])
    verified = phase_i_browser.bundle
    display({
        'run_id': phase_i_browser.manifest.run_id,
        'fold_plan': verified.fold_plan.to_dict(),
        'completion_index': verified.completion_index.to_dict(),
        'expected_primary_trial_ids': phase_i_browser.manifest.expected_primary_trial_ids,
        'finalist_freeze': None if verified.finalist_freeze is None else verified.finalist_freeze.to_dict(),
        'holdout_open_audits': [audit.to_dict() for audit in verified.holdout_open_audits],
        'baseline_holdout': None if verified.baseline_holdout is None else verified.baseline_holdout.to_dict(),
        'finalist_holdout': None if verified.finalist_holdout is None else verified.finalist_holdout.to_dict(),
        'recommendation': phase_i_browser.recommendation.to_dict(),
    })
    display(artifact_trial_table)

## 12. Stage-Specific Parameter Sensitivity

In [ ]:
if phase_i_browser is not None:
    sensitivity_rows = artifact_trial_rows(phase_i_browser.trials)
    for stage in sorted({row.stage for row in sensitivity_rows}):
        stage_rows = tuple(
            row for row in sensitivity_rows
            if row.stage == stage and row.validation_only and row.primary_metric_value is not None
        )
        if stage_rows:
            metric = phase_i_browser.manifest.objective_specs[stage].primary_metric
            display({
                'stage': stage,
                'metric': metric,
                'validation_only_rows': [record_to_dict(row) for row in stage_rows],
            })
        else:
            display({'stage': stage, 'status': 'UNAVAILABLE', 'reason': 'no defined validation metric values'})
# Holdout evidence never reranks or tunes this grid.

## 13. Multi-Asset Comparison

Disabled by default. Compare only declared comparable structural samples. Never compare PnL.

In [ ]:
def compare_replay_summaries(replays):
    audit = audit_cross_asset_comparability(replays, policy=COMPARISON_POLICY)
    if not audit.comparable:
        return audit, pd.DataFrame()
    comparison = build_cross_asset_comparison(replays, policy=COMPARISON_POLICY)
    return comparison.audit, pd.DataFrame([record_to_dict(row) for row in comparison.rows])

if RUN_MULTI_ASSET_COMPARISON:
    comparison_audit, comparison_table = compare_replay_summaries(MULTI_ASSET_REPLAYS)
    display(record_to_dict(comparison_audit))
    display(comparison_table)

## 14. Performance Diagnostics

In [ ]:
performance = {'bars_processed': dataset.row_count, **replay.runtime_diagnostics}
display(performance)

## 15. Export and Reproducibility

In [ ]:
if EXPORT_ARTIFACTS:
    exported = export_research_artifacts(
        output_root=OUTPUT_ROOT,
        replay=replay,
        tables={
            'candidate_rows': candidate_rows(replay),
            'provider_audits': provider_audit_rows(replay),
            'families': family_rows(final_snapshot),
            'rails': member_rail_rows(final_snapshot),
            'corridors': corridor_rows(final_snapshot),
            'zones': interaction_zone_rows(final_snapshot),
            'transitions': transition_rows(final_snapshot),
            'observations': observation_rows(final_snapshot),
            'events': event_rows(final_snapshot),
            'event_transitions': event_transition_rows(final_snapshot),
            'source_groups': source_group_audit_rows(final_snapshot),
            'structural_outcomes': structural_outcome_rows(replay),
        },
        selected_position=dataset.row_count - 1,
        mtf_snapshot=globals().get('mtf_snapshot'),
        phase_i_browser=phase_i_browser,
    )
    display(exported)